In [2]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("spark_sql")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

df_green = spark.read.parquet("data/pq/green/*/*")
df_yellow = spark.read.parquet("data/pq/yellow/*/*")
print(len(df_green.columns), "green columns", len(df_yellow.columns), "yellow columns")



Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/16 19:04:03 WARN Utils: Your hostname, DESKTOP-8DK3P4V, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/16 19:04:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/16 19:04:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/16 19:04:13 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/pq/green/*/*.
java.io.FileNotFoundException: File data/pq/green/*/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.

20 green columns 18 yellow columns


In [3]:
df_green = (
    df_green
    .withColumnRenamed("lpep_pickup_datetime", "pickup_datetime")
    .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime")
)
df_yellow = (
    df_yellow
    .withColumnRenamed("tpep_pickup_datetime", "pickup_datetime")
    .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")
)

In [4]:
yellow_columns = set(df_yellow.columns)
common_columns = [c for c in df_green.columns if c in yellow_columns]
print(len(common_columns), common_columns)

18 ['VendorID', 'pickup_datetime', 'dropoff_datetime', 'store_and_fwd_flag', 'RatecodeID', 'PULocationID', 'DOLocationID', 'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'payment_type', 'congestion_surcharge']


In [5]:
df_green_sel = df_green.select(common_columns).withColumn("service_type", F.lit("green"))
df_yellow_sel = df_yellow.select(common_columns).withColumn("service_type", F.lit("yellow"))

df_trips = df_green_sel.unionAll(df_yellow_sel)

In [6]:
df_trips.groupBy("service_type").count().show()

+------------+--------+
|service_type|   count|
+------------+--------+
|       green| 2304517|
|      yellow|39649199|
+------------+--------+



In [7]:
df_trips.filter(F.col("service_type") == "yellow").show(10)

+--------+-------------------+-------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------------------+------------+------------+--------------------+------------+
|VendorID|    pickup_datetime|   dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|payment_type|congestion_surcharge|service_type|
+--------+-------------------+-------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------------------+------------+------------+--------------------+------------+
|       1|2020-01-01 19:45:57|2020-01-01 20:06:13|                 N|         1|         162|          87|              1|          6.1|       21.0|  2.5|    0.5|      4.85|         0.0|       

In [8]:
df_trips.createOrReplaceTempView("trips_data")

spark.sql("""
    SELECT service_type, COUNT(*) AS trips
    FROM trips_data
    GROUP BY service_type
""").show()

+------------+--------+
|service_type|   trips|
+------------+--------+
|       green| 2304517|
|      yellow|39649199|
+------------+--------+



In [9]:
sql_version = spark.sql("""
    SELECT service_type, COUNT(*) AS trips
    FROM trips_data
    GROUP BY service_type
""")
df_version = df_trips.groupBy("service_type").agg(F.count("*").alias("trips"))

print("=== SQL ===")
sql_version.explain()
print("=== DataFrame ===")
df_version.explain()

=== SQL ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[service_type#44], functions=[count(1)])
   +- Exchange hashpartitioning(service_type#44, 200), ENSURE_REQUIREMENTS, [plan_id=202]
      +- HashAggregate(keys=[service_type#44], functions=[partial_count(1)])
         +- Union
            :- Project [green AS service_type#44]
            :  +- FileScan parquet [] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(19 paths)[file:/home/velpu/projects/data-engineering-project1/06-batch/data/pq/..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<>
            +- Project [yellow AS service_type#45]
               +- FileScan parquet [] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(19 paths)[file:/home/velpu/projects/data-engineering-project1/06-batch/data/pq/..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<>


=== DataFrame ===
== Physical Plan ==
AdaptiveSparkPlan isF

In [10]:
df_revenue = spark.sql("""
            SELECT
        PULocationID AS revenue_zone,
        date_trunc('month', pickup_datetime) AS revenue_month,
        service_type,

        SUM(fare_amount) AS revenue_monthly_fare,
        SUM(extra) AS revenue_monthly_extra,
        SUM(mta_tax) AS revenue_monthly_mta_tax,
        SUM(tip_amount) AS revenue_monthly_tip_amount,
        SUM(tolls_amount) AS revenue_monthly_tolls_amount,
        SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
        SUM(total_amount) AS revenue_monthly_total_amount,
        SUM(congestion_surcharge) AS revenue_monthly_congestion_surcharge,

        AVG(passenger_count) AS avg_monthly_passenger_count,
        AVG(trip_distance) AS avg_monthly_trip_distance
    FROM trips_data
    GROUP BY 1, 2, 3
           """)

df_revenue.show()

+------------+-------------------+------------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+---------------------------+-------------------------+
|revenue_zone|      revenue_month|service_type|revenue_monthly_fare|revenue_monthly_extra|revenue_monthly_mta_tax|revenue_monthly_tip_amount|revenue_monthly_tolls_amount|revenue_monthly_improvement_surcharge|revenue_monthly_total_amount|revenue_monthly_congestion_surcharge|avg_monthly_passenger_count|avg_monthly_trip_distance|
+------------+-------------------+------------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+---------------------------+-------------------------+
|         218

In [11]:
df_revenue.coalesce(1).write.mode("overwrite").parquet("data/report/revenue/")

In [12]:
report = spark.read.parquet("data/report/revenue/")
print("Report rows:", report.count())

report.createOrReplaceTempView("revenue")
spark.sql("""
    SELECT service_type, revenue_month, ROUND(SUM(revenue_monthly_total_amount), 2) AS total_revenue
    FROM revenue
    WHERE revenue_month >= '2020-01-01' AND revenue_month < '2020-07-01'
    GROUP BY service_type, revenue_month
    ORDER BY service_type, revenue_month
""").show(12)

Report rows: 9895
+------------+-------------------+--------------+
|service_type|      revenue_month| total_revenue|
+------------+-------------------+--------------+
|       green|2020-01-01 00:00:00|    8461729.88|
|       green|2020-02-01 00:00:00|    7376880.78|
|       green|2020-03-01 00:00:00|    3733663.56|
|       green|2020-04-01 00:00:00|      700958.6|
|       green|2020-05-01 00:00:00|    1225120.75|
|       green|2020-06-01 00:00:00|    1450810.72|
|      yellow|2020-01-01 00:00:00|1.1953872193E8|
|      yellow|2020-02-01 00:00:00| 1.171142335E8|
|      yellow|2020-03-01 00:00:00| 5.665517399E7|
|      yellow|2020-04-01 00:00:00|    3903167.83|
|      yellow|2020-05-01 00:00:00|    6854105.06|
|      yellow|2020-06-01 00:00:00| 1.031864323E7|
+------------+-------------------+--------------+



In [13]:
spark.stop()

In [14]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("groupby")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

df_green = spark.read.parquet("data/pq/green/*/*")
df_green.createOrReplaceTempView("green")
print("Input partitions:", df_green.rdd.getNumPartitions())

26/11/16 19:06:03 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/pq/green/*/*.
java.io.FileNotFoundException: File data/pq/green/*/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:401)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:209)
	at org.apache.spark.sql.catalyst.analysis.ResolveData

Input partitions: 3


In [15]:
spark.sparkContext.setJobDescription("green hourly revenue")

df_green_revenue = spark.sql("""
    SELECT
        date_trunc('hour', lpep_pickup_datetime) AS hour,
        PULocationID AS zone,
        SUM(total_amount) AS amount,
        COUNT(1) AS number_records
    FROM green
    WHERE lpep_pickup_datetime >= '2020-01-01 00:00:00'
    GROUP BY 1, 2
""")

df_green_revenue.write.mode("overwrite").parquet("data/report/revenue/green_hourly")

In [16]:
df_green_revenue.show()

+-------------------+----+------------------+--------------+
|               hour|zone|            amount|number_records|
+-------------------+----+------------------+--------------+
|2020-01-06 07:00:00| 241|            169.16|             6|
|2020-01-19 19:00:00|  41| 457.2400000000003|            43|
|2020-01-23 23:00:00| 129| 304.1300000000001|            22|
|2020-01-28 15:00:00| 129|            208.23|             9|
|2020-01-02 08:00:00|  74|1187.4199999999987|            96|
|2020-01-06 16:00:00| 129|250.42000000000002|            15|
|2020-01-07 10:00:00| 171|             38.98|             2|
|2020-01-09 15:00:00| 260| 242.6800000000001|            19|
|2020-01-16 20:00:00|  82| 424.1900000000001|            27|
|2020-01-21 16:00:00|  33| 917.3599999999994|            38|
|2020-01-08 17:00:00| 140|            313.81|             8|
|2020-01-22 08:00:00| 244| 285.3999999999999|            14|
|2020-01-30 14:00:00|  74| 738.6799999999996|            49|
|2020-01-03 07:00:00|  7

In [1]:
spark.stop()

NameError: name 'spark' is not defined